<a href="https://colab.research.google.com/github/nolanlwin/mech-interp-coding-llms/blob/main/Qwen2_5_1_5B_Instruct.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -U transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 65.7 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0


## Local Inference on GPU
Model page: https://huggingface.co/Qwen/Qwen2.5-1.5B-Instruct

⚠️ If the generated code snippets do not work, please open an issue on either the [model repo](https://huggingface.co/Qwen/Qwen2.5-1.5B-Instruct)
			and/or on [huggingface.js](https://github.com/huggingface/huggingface.js/blob/main/packages/tasks/src/model-libraries-snippets.ts) 🙏

In [2]:
# Use a pipeline as a high-level helper
from transformers import pipeline

pipe = pipeline("text-generation", model="Qwen/Qwen2.5-1.5B-Instruct")
messages = [
    {"role": "user", "content": "Who are you?"},
]
pipe(messages)

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[{'generated_text': [{'role': 'user', 'content': 'Who are you?'},
   {'role': 'assistant',
    'content': 'I am Qwen, an AI language model developed by Alibaba Cloud. I was designed to assist with various tasks such as answering questions, providing information, and having conversations on a wide range of topics. Is there anything specific you would like to know or discuss?'}]}]

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-1.5B-Instruct")
model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen2.5-1.5B-Instruct")
messages = [
    {"role": "user", "content": "Generate 25 Python coding examples that demonstrate the use of boolean flags. Each example should be self-contained and clearly illustrate a different scenario where a boolean flag is useful, such as controlling program flow, indicating state, or validating input. For each example, include a brief comment explaining the purpose of the boolean flag."}
]
inputs = tokenizer.apply_chat_template(
	messages,
	add_generation_prompt=True,
	tokenize=True,
	return_dict=True,
	return_tensors="pt",
).to(model.device)

outputs = model.generate(**inputs, max_new_tokens=4000) # Increased max_new_tokens to accommodate 25 examples
generated_text = tokenizer.decode(outputs[0][inputs["input_ids"].shape[-1]:])
print("Generated examples:\n", generated_text)

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

In [7]:
import re
import os

def extract_and_save_code_examples(generated_text, output_dir="code_samples"):
    os.makedirs(output_dir, exist_ok=True)

    # First, extract the single large Python code block that contains all examples
    main_code_block_match = re.search(r'```python\n(.*?)```', generated_text, re.DOTALL)

    if not main_code_block_match:
        print("No main Python code block found in the generated text.")
        return

    full_code_content = main_code_block_match.group(1).strip()

    # Now, split this full_code_content into individual examples
    # Each example starts with '# Example X:'
    # This regex looks for '# Example X:', then captures everything until the next '# Example Y:' or the end of the string.
    individual_examples = re.findall(r'(# Example \d+:\s*.*?)(?=\n# Example \d+:|\Z)', full_code_content, re.DOTALL)

    if not individual_examples:
        print("No individual examples found within the main code block.")
        return

    print(f"Found {len(individual_examples)} code examples. Saving to '{output_dir}'...")

    for i, example_content in enumerate(individual_examples):
        # Extract the example number from the first line of the example content
        example_num_match = re.search(r'# Example (\d+):', example_content)
        if example_num_match:
            example_number = int(example_num_match.group(1))
        else:
            example_number = i + 1 # Fallback if number not found in comment

        filename = os.path.join(output_dir, f"example_{example_number:02d}.py")
        with open(filename, "w") as f:
            f.write(example_content.strip())
        print(f"Saved {filename}")

# Call the function with the generated text
extract_and_save_code_examples(generated_text)


Found 12 code examples. Saving to 'code_samples'...
Saved code_samples/example_01.py
Saved code_samples/example_02.py
Saved code_samples/example_03.py
Saved code_samples/example_04.py
Saved code_samples/example_05.py
Saved code_samples/example_06.py
Saved code_samples/example_07.py
Saved code_samples/example_08.py
Saved code_samples/example_09.py
Saved code_samples/example_10.py
Saved code_samples/example_11.py
Saved code_samples/example_12.py
